# Module 1: Scraping Setup and Execution

## Objective

Scrape country-level military metrics from the URLs provided in:

```
links_for_military_data.txt
```

Extract key military data and store it in a CSV file.

---

## Tasks

### 1. Load URLs

* Read all URLs from `links_for_military_data.txt`
* Each URL corresponds to one country

### 2. Scrape Military Metrics

For each country page, extract:

* Country Name
* Total Aircraft
* Total Tanks
* Defense Budget
* Naval Assets
* Active Personnel

### 3. Store Raw Data

* Save extracted data into:

```
military_raw_data.csv
```


---

* At least 95% of URLs successfully scraped
* Data collected for 140+ countries
* Military metrics correctly parsed
* CSV file properly structured

---

## Tools

* requests
* BeautifulSoup
* pandas

---
 A structured CSV file containing military metrics for 140+ countries.


In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = 'https://www.globalfirepower.com/countries-listing.php'

OTHER_SOURCES = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
    'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
    'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
    'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
    'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
    'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
    'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
    'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
    'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
    'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
    'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
    'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
    'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
    'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
    'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
    'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
    'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
    'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
    'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
    'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
    'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}


def get_soup(url):
    response = requests.get(
        url,
        headers={'User-Agent': 'Mozilla/5.0'},
        timeout=20
    )
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def get_country_list():
    soup = get_soup(BASE_URL)

    country_spans = soup.select('div.longFormName > span')

    countries = []
    for span in country_spans:
        country = span.text.strip()
        if country:
            countries.append(country)

    return pd.DataFrame({'Country': countries})


def scrape_metric_page(url):
    soup = get_soup(url)

    records = soup.select('div.recordsetContainer')
    metric_data = {}

    for record in records:
        country_span = record.select_one('div.longFormName > span')
        value_span = record.select_one('div.valueContainer span span')

        if country_span and value_span:
            country = country_span.text.strip()
            value = value_span.text.strip()
            metric_data[country] = value

    return metric_data


def build_dataset():
    master_df = get_country_list()
    print(f"Countries loaded: {len(master_df)}")
    print(f"Total metrics to scrape: {len(OTHER_SOURCES)}")

    for i, (url, column_name) in enumerate(OTHER_SOURCES.items(), 1):
        print(f"[{i}/{len(OTHER_SOURCES)}] Scraping: {column_name}")

        try:
            metric_dict = scrape_metric_page(url)

            master_df[column_name] = master_df['Country'].map(metric_dict)

            time.sleep(2)

        except Exception as e:
            print(f"  ❌ Error scraping {column_name}: {e}")
            continue

    return master_df

if __name__ == "__main__":
    print("="*60)
    print("Starting Global Firepower Data Scraping")
    print("="*60)

    df = build_dataset()
    print("\n" + "="*60)
    print(f"✅ Scraping complete!")
    print(f"Shape: {df.shape[0]} countries × {df.shape[1]} columns")
    print("="*60)

    print("\nFirst 5 rows:")
    print(df.head())

    df.to_csv('global_firepower_data.csv', index=False)
    print("\n✅ Data saved to 'global_firepower_data.csv'")

    print(f"\nTotal columns: {len(df.columns)}")
    print("\nAll columns:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i}. {col}")

Starting Global Firepower Data Scraping
Countries loaded: 145
Total metrics to scrape: 54
[1/54] Scraping: total_population
[2/54] Scraping: total_military_manpower
[3/54] Scraping: fit_for_service
[4/54] Scraping: population_reaching_military_age_annually
[5/54] Scraping: active_personnel
[6/54] Scraping: reserve_personnel
[7/54] Scraping: paramilitary
[8/54] Scraping: total_military_aircraft
[9/54] Scraping: fighter_aircraft
[10/54] Scraping: attack_aircraft
[11/54] Scraping: transport_aircraft
[12/54] Scraping: trainer_aircraft
[13/54] Scraping: special_mission_aircraft
[14/54] Scraping: tanker_aircraft
[15/54] Scraping: total_military_helicopters
[16/54] Scraping: attack_helicopters
[17/54] Scraping: tanks
[18/54] Scraping: armored_fighting_vehicles
[19/54] Scraping: self_propelled_artillery
[20/54] Scraping: towed_artillery
[21/54] Scraping: rocket_projectors
[22/54] Scraping: total_naval_fleet
[23/54] Scraping: total_naval_fleet_tonnage_mt
[24/54] Scraping: aircraft_carriers
[25/

# Module 2: Data Cleaning and Structuring

## Objective

Clean and structure the scraped military dataset so it can be directly used in Power BI for visualization and analysis.

---

## Tasks

### 1. Text Cleaning

* Remove commas (,), percentage signs (%), plus signs (+)
* Remove currency symbols and special characters
* Trim extra spaces
* Ensure consistent formatting

---

### 2. Convert to Numeric Format

* Convert aircraft, tanks, personnel, budget, and other metrics to numeric types
* Ensure no text values remain in numeric columns
* Handle conversion errors safely

---

### 3. Standardize Column Names

Rename columns using consistent lowercase and underscore format. Example:

* Country → country
* Total Aircraft → total_aircraft
* Active Personnel → active_personnel
* Defense Budget → defense_budget

---

### 4. Handle Missing or Null Values

* Identify missing values
* Replace with appropriate values (0 or median where applicable)
* Ensure final dataset has less than 2% missing/null values

---

### 5. Save Clean Dataset

Save the final cleaned dataset as:

```
military_cleaned.csv
```

This file will be directly imported into Power BI.


* Less than 2% missing/null values after cleaning
* All numeric columns properly formatted
* No structural errors in dataset
* Compatible for direct import into Power BI

---

A fully cleaned and structured dataset ready for Power BI dashboard creation.


In [ ]:
df = pd.read_csv('global_firepower_data.csv')

print(f"Shape: {df.shape}")
print("\n" + "="*60)

print("\nData Types:")
print(df.dtypes)
print("\n" + "="*60)

print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*60)

print("\nFirst 3 rows:")
print(df.head(3))
print("\n" + "="*60)

print("\nSample values from numeric columns:")
print(df[['total_population', 'active_personnel', 'tanks', 'defense_budget_usd']].head())

Shape: (145, 55)


Data Types:
Country                                      object
total_population                             object
total_military_manpower                      object
fit_for_service                              object
population_reaching_military_age_annually    object
active_personnel                             object
reserve_personnel                            object
paramilitary                                 object
total_military_aircraft                      object
fighter_aircraft                             object
attack_aircraft                               int64
transport_aircraft                            int64
trainer_aircraft                             object
special_mission_aircraft                      int64
tanker_aircraft                               int64
total_military_helicopters                   object
attack_helicopters                           object
tanks                                        object
armored_fighting_vehicles        

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("global_firepower_data.csv")
df.head()


,Country,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,United States,"341,963,408","150,463,900","124,816,644","4,445,524","1,328,000","799,500",0,"13,043","1,790",...,"1,029,000,000,000 \t\t\...","914,301,000,000 \t\t\t\...","13,402,000,000,000 \t\t...","548,849,000 \t\t\t\t\t\...","476,044,000 \t\t\t\t\t\...","248,941,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\r\n...","12,002 \t\t\t\t\t\t\r\n...","41,009 \t\t\t\t\t\t\r\n..."
1,Russia,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000","250,000","4,292",833,...,"617,830,000,000 \t\t\t\...","472,239,000,000 \t\t\t\...","47,805,000,000,000 \t\t...","508,190,000 \t\t\t\t\t\...","310,958,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\r\n...","22,407 \t\t\t\t\t\t\r\n...","102,000 \t\t\t\t\t\t\r\..."
2,China,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000","625,000","3,309","1,212",...,"225,341,000,000 \t\t\t\...","366,160,000,000 \t\t\t\...","6,654,000,000,000 \t\t\...","4,827,000,000 \t\t\t\t\...","5,313,000,000 \t\t\t\t\...","143,197,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\r\n...","22,457 \t\t\t\t\t\t\r\n...","27,700 \t\t\t\t\t\t\r\n..."
3,India,"1,409,128,296","662,290,299","522,786,598","23,955,181","1,455,550","1,155,000","2,527,000","2,229",513,...,"33,170,000,000 \t\t\t\t...","58,867,000,000 \t\t\t\t...","1,381,000,000,000 \t\t\...","985,671,000 \t\t\t\t\t\...","1,200,000,000 \t\t\t\t\...","111,052,000,000 \t\t\t\...","3,287,263 \t\t\t\t\t\t\...","7,000 \t\t\t\t\t\t\r\n\...","13,888 \t\t\t\t\t\t\r\n...","14,500 \t\t\t\t\t\t\r\n..."
4,South Korea,"52,081,799","26,040,900","21,353,538","416,654","600,000","3,100,000","120,000","1,592",315,...,"55,127,000 \t\t\t\t\t\t...","59,480,000,000 \t\t\t\t...","7,079,000,000 \t\t\t\t\...","15,595,000 \t\t\t\t\t\t...","136,413,000 \t\t\t\t\t\...","326,000,000 \t\t\t\t\t\...","99,720 \t\t\t\t\t\t\r\n...","2,413 \t\t\t\t\t\t\r\n\...",237 \t\t\t\t\t\t\r\n\t\...,"1,600 \t\t\t\t\t\t\r\n\..."


In [ ]:
df.dtypes


,0
Country,object
total_population,object
total_military_manpower,object
fit_for_service,object
population_reaching_military_age_annually,object
active_personnel,object
reserve_personnel,object
paramilitary,object
total_military_aircraft,object
fighter_aircraft,object


In [ ]:
def clean_numeric_column(col):
    return (
        col.astype(str)
        .str.replace(",", "", regex=True)
        .str.replace("$", "", regex=False)
        .str.replace("km", "", regex=False)
        .str.replace("bbl", "", regex=False)
        .str.replace("Cu.M", "", regex=False)
        .str.replace("mt", "", regex=False)
        .str.replace("MT", "", regex=False)
        .str.replace(" ", "", regex=False)
        .replace(["", "N/A", "NA", "-", "None"], np.nan)
    )


In [ ]:
numeric_cols = [col for col in df.columns if col != "Country"]

for col in numeric_cols:
    df[col] = clean_numeric_column(df[col])


In [ ]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(0)


In [ ]:
df[numeric_cols] = df[numeric_cols].clip(lower=0)


In [ ]:
df.dtypes


,0
Country,object
total_population,int64
total_military_manpower,int64
fit_for_service,int64
population_reaching_military_age_annually,int64
active_personnel,int64
reserve_personnel,int64
paramilitary,int64
total_military_aircraft,int64
fighter_aircraft,int64


In [ ]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
total_population,145.0,5.456547e+07,1.702420e+08,3.640360e+05,5.650957e+06,1.469705e+07,3.879481e+07,1.415043e+09
total_military_manpower,145.0,2.581125e+07,8.583942e+07,8.372800e+04,2.392390e+06,6.351857e+06,1.794604e+07,7.641234e+08
fit_for_service,145.0,2.019565e+07,6.928101e+07,5.023700e+04,1.912981e+06,4.096295e+06,1.403738e+07,6.268642e+08
population_reaching_military_age_annually,145.0,8.546933e+05,2.667628e+06,1.820000e+03,7.199100e+04,2.048300e+05,6.250060e+05,2.395518e+07
active_personnel,145.0,1.538843e+05,2.968488e+05,0.000000e+00,1.840000e+04,5.000000e+04,1.620000e+05,2.035000e+06
reserve_personnel,145.0,2.018092e+05,5.759480e+05,0.000000e+00,0.000000e+00,2.600000e+04,1.300000e+05,5.000000e+06
paramilitary,145.0,1.213070e+05,6.096424e+05,0.000000e+00,2.000000e+03,1.250000e+04,5.500000e+04,6.800000e+06
total_military_aircraft,145.0,3.595793e+02,1.191107e+03,0.000000e+00,2.500000e+01,9.700000e+01,2.600000e+02,1.304300e+04
fighter_aircraft,145.0,7.048276e+01,2.027830e+02,0.000000e+00,0.000000e+00,1.100000e+01,4.500000e+01,1.790000e+03
attack_aircraft,145.0,2.591724e+01,9.886119e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.700000e+01,8.890000e+02


In [ ]:
df.to_csv("clean_military_dataset.csv", index=False)


In [ ]:
!pip install pycountry scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 53.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import pycountry
from sklearn.preprocessing import MinMaxScaler


In [ ]:
df = pd.read_csv("clean_military_dataset.csv")


In [ ]:
def get_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except:
        return "UNK"

# Fix common naming issues
name_fixes = {
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Iran": "Iran, Islamic Republic of",
    "Russia": "Russian Federation",
    "Syria": "Syrian Arab Republic",
    "Vietnam": "Viet Nam"
}

df['Country_fixed'] = df['Country'].replace(name_fixes)
df['ISO_Code'] = df['Country_fixed'].apply(get_iso3)


In [ ]:
df['Manpower_Potential_Rank'] = df['total_population'].rank(
    ascending=False,
    method='dense'
).astype(int)


In [ ]:
# Manpower Index
df['Manpower_Index'] = (
    df['active_personnel'] +
    df['reserve_personnel'] +
    df['fit_for_service']
)

# Land Power Index
df['Land_Power_Index'] = (
    df['tanks'] +
    df['armored_fighting_vehicles'] +
    df['self_propelled_artillery'] +
    df['towed_artillery'] +
    df['rocket_projectors']
)

# Air Power Index
df['Air_Power_Index'] = (
    df['total_military_aircraft'] +
    df['attack_helicopters']
)

# Naval Power Index
df['Naval_Power_Index'] = (
    df['total_naval_fleet'] +
    df['submarines'] +
    df['aircraft_carriers'] +
    df['destroyers'] +
    df['frigates']
)

# Economic Power Index
df['Economic_Power_Index'] = (
    df['defense_budget_usd'] +
    df['purchasing_power_parity_usd'] +
    df['foreign_exchange_and_gold_reserves_usd']
)


In [ ]:
scaler = MinMaxScaler()

index_cols = [
    'Manpower_Index',
    'Land_Power_Index',
    'Air_Power_Index',
    'Naval_Power_Index',
    'Economic_Power_Index'
]

df[index_cols] = scaler.fit_transform(df[index_cols])


In [ ]:
def assign_power_category(row):
    avg_score = row[index_cols].mean()
    if avg_score >= 0.70:
        return "High Power"
    elif avg_score >= 0.45:
        return "Medium Power"
    else:
        return "Low Power"

df['Power_Category'] = df.apply(assign_power_category, axis=1)


In [ ]:
final_df = df[[
    'ISO_Code',
    'Country',
    'Manpower_Potential_Rank',
    'Manpower_Index',
    'Land_Power_Index',
    'Air_Power_Index',
    'Naval_Power_Index',
    'Economic_Power_Index',
    'Power_Category'
]]

final_df.head()


,ISO_Code,Country,Manpower_Potential_Rank,Manpower_Index,Land_Power_Index,Air_Power_Index,Naval_Power_Index,Economic_Power_Index,Power_Category
0,USA,United States,3,0.201624,1.000000,1.000000,0.657923,0.753466,High Power
1,RUS,Russia,9,0.078586,0.385600,0.345247,0.551913,0.187001,Low Power
2,CHN,China,1,1.000000,0.395880,0.255607,1.000000,1.000000,High Power
3,IND,India,2,0.834733,0.393567,0.164400,0.371585,0.395014,Low Power
4,KOR,South Korea,28,0.039728,0.173234,0.121253,0.304918,0.088055,Low Power


In [ ]:
final_df = final_df.sort_values(
    by='Manpower_Potential_Rank',
    ascending=True
).reset_index(drop=True)


In [ ]:
final_df.head(10)


,S_No,ISO_Code,Country,Manpower_Potential_Rank,Manpower_Index,Land_Power_Index,Air_Power_Index,Naval_Power_Index,Economic_Power_Index,Power_Category
0,1,CHN,China,1,1.000000,0.395880,0.255607,1.000000,1.000000,High Power
1,2,IND,India,2,0.834733,0.393567,0.164400,0.371585,0.395014,Low Power
2,3,USA,United States,3,0.201624,1.000000,1.000000,0.657923,0.753466,High Power
3,4,IDN,Indonesia,4,0.183275,0.053375,0.033749,0.373770,0.116125,Low Power
4,5,PAK,Pakistan,5,0.138168,0.060019,0.103667,0.150820,0.039001,Low Power
5,6,NGA,Nigeria,6,0.143983,0.024118,0.012674,0.146448,0.037602,Low Power
6,7,BRA,Brazil,7,0.141939,0.058222,0.036525,0.080874,0.125691,Low Power
7,8,BGD,Bangladesh,8,0.105253,0.031405,0.015237,0.138798,0.041029,Low Power
8,9,RUS,Russia,9,0.078586,0.385600,0.345247,0.551913,0.187001,Low Power
9,10,MEX,Mexico,10,0.079879,0.049278,0.030829,0.189071,0.088418,Low Power


In [ ]:
df.columns

Index(['Country', 'total_population', 'total_military_manpower',
       'fit_for_service', 'population_reaching_military_age_annually',
       'active_personnel', 'reserve_personnel', 'paramilitary',
       'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft',
       'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft',
       'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters',
       'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
       'major_ports_and_terminals', 'total_merchant_

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('final_unified_military_power_dataset.csv')
df.head(100)

,Country,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,...,waterway_coverage_km,Country_fixed,ISO_Code,Manpower_Potential_Rank,Manpower_Index,Land_Power_Index,Air_Power_Index,Naval_Power_Index,Economic_Power_Index,Power_Category
0,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,41009,United States,USA,3,0.201624,1.000000,1.000000,0.657923,0.753466,High Power
1,Russia,140820810,69002197,46189226,1267387,1320000,2000000,250000,4292,833,...,102000,Russian Federation,RUS,9,0.078586,0.385600,0.345247,0.551913,0.187001,Low Power
2,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,27700,China,CHN,1,1.000000,0.395880,0.255607,1.000000,1.000000,High Power
3,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,14500,India,IND,2,0.834733,0.393567,0.164400,0.371585,0.395014,Low Power
4,South Korea,52081799,26040900,21353538,416654,600000,3100000,120000,1592,315,...,1600,"Korea, Republic of",KOR,28,0.039728,0.173234,0.121253,0.304918,0.088055,Low Power
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Slovenia,2097893,1006989,826570,22867,7300,26000,5000,39,0,...,710,Slovenia,SVN,135,0.001286,0.003548,0.002777,0.002186,0.002849,Low Power
96,Ireland,5233461,2407392,1999182,57568,7765,1700,0,22,0,...,956,Ireland,IRL,115,0.003112,0.004080,0.001566,0.008743,0.017663,Low Power
97,Mongolia,3281676,1870555,1552233,62352,35000,135000,50000,7,0,...,580,Mongolia,MNG,125,0.002657,0.008836,0.000498,0.000000,0.001587,Low Power
98,Latvia,1801246,810561,648449,16211,17250,36000,12500,7,0,...,300,Latvia,LVA,137,0.001035,0.004538,0.000498,0.019672,0.002046,Low Power


In [ ]:
import pandas as pd

df = pd.read_csv("final_unified_military_power_dataset.csv")


Standardize the Column Names


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w_]", "", regex=True)
)


Clean numeric columns (symbols → numbers)

In [ ]:
numeric_cols = df.columns.drop(["country", "iso_code", "power_category"])

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace("%", "")
        .str.replace("+", "")
        .str.replace("$", "")
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
required_cols = [
    "iso_code",
    "country",
    "manpower_potential_rank",
    "overall_power_rank",
    "population",
    "military_budget",
    "gdp",
    "total_military_assets"
]

missing = [c for c in required_cols if c not in df.columns]
missing


['overall_power_rank',
 'population',
 'military_budget',
 'gdp',
 'total_military_assets']

In [ ]:
df.columns

Index(['country', 'total_population', 'total_military_manpower',
       'fit_for_service', 'population_reaching_military_age_annually',
       'active_personnel', 'reserve_personnel', 'paramilitary',
       'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft',
       'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft',
       'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters',
       'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
       'major_ports_and_terminals', 'total_merchant_

In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False).head(50)


,0
country_fixed,100.0
country,0.0
total_military_manpower,0.0
total_population,0.0
population_reaching_military_age_annually,0.0
active_personnel,0.0
reserve_personnel,0.0
fit_for_service,0.0
total_military_aircraft,0.0
fighter_aircraft,0.0


In [ ]:
df = df.drop(columns=["country_fixed"])


In [ ]:
(df.isna().mean() * 100).max()


0.0

In [ ]:
df.to_csv("military_cleaned.csv", index=False)


In [ ]:
df=pd.read_csv('military_cleaned.csv')
df.head()

,country,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,...,border_coverage_km,waterway_coverage_km,iso_code,manpower_potential_rank,manpower_index,land_power_index,air_power_index,naval_power_index,economic_power_index,power_category
0,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,12002,41009,USA,3,0.201624,1.000000,1.000000,0.657923,0.753466,High Power
1,Russia,140820810,69002197,46189226,1267387,1320000,2000000,250000,4292,833,...,22407,102000,RUS,9,0.078586,0.385600,0.345247,0.551913,0.187001,Low Power
2,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,22457,27700,CHN,1,1.000000,0.395880,0.255607,1.000000,1.000000,High Power
3,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,13888,14500,IND,2,0.834733,0.393567,0.164400,0.371585,0.395014,Low Power
4,South Korea,52081799,26040900,21353538,416654,600000,3100000,120000,1592,315,...,237,1600,KOR,28,0.039728,0.173234,0.121253,0.304918,0.088055,Low Power


# Module 3: KPI Engineering

## Objective

Create derived Key Performance Indicators (KPIs) from the cleaned military dataset for analytical use in Power BI.

---

## 1. Total Military Assets

**Formula:**

```
total_military_assets =
    tanks
  + armored_fighting_vehicles
  + total_military_aircraft
  + total_naval_fleet
```

**Description:**
Represents the total physical military strength based on major equipment categories.

---

## 2. Assets per Capita

**Formula:**

```
assets_per_capita =
    total_military_assets
  / total_population
```

**Description:**
Measures military assets relative to population size.

---

## 3. Budget to GDP Ratio

**Formula:**

```
budget_to_gdp_ratio =
    defense_budget_usd
  / purchasing_power_parity_usd
```

**Description:**
Indicates the proportion of economic strength allocated to defense.

---

## 4. Power Index Rank Gap

**Formula:**

```
power_index_rank_gap =
    manpower_potential_rank
  - overall_power_rank
```

**Description:**
Shows the difference between manpower strength ranking and overall military power ranking.

---

## Expected Output

These KPIs should be added as new calculated columns in the dataset and included in:

```
military_cleaned.csv
```

The updated dataset will then be used in Power BI for dashboard development.


In [ ]:
import pandas as pd

# Load cleaned dataset from Module 2
df = pd.read_csv("military_cleaned.csv")
df.head()


,country,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,...,border_coverage_km,waterway_coverage_km,iso_code,manpower_potential_rank,manpower_index,land_power_index,air_power_index,naval_power_index,economic_power_index,power_category
0,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,12002,41009,USA,3,0.201624,1.000000,1.000000,0.657923,0.753466,High Power
1,Russia,140820810,69002197,46189226,1267387,1320000,2000000,250000,4292,833,...,22407,102000,RUS,9,0.078586,0.385600,0.345247,0.551913,0.187001,Low Power
2,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,22457,27700,CHN,1,1.000000,0.395880,0.255607,1.000000,1.000000,High Power
3,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,13888,14500,IND,2,0.834733,0.393567,0.164400,0.371585,0.395014,Low Power
4,South Korea,52081799,26040900,21353538,416654,600000,3100000,120000,1592,315,...,237,1600,KOR,28,0.039728,0.173234,0.121253,0.304918,0.088055,Low Power


KPI Engineering

In [ ]:
# Total military assets
df["total_military_assets"] = (
    df["tanks"]
    + df["armored_fighting_vehicles"]
    + df["total_military_aircraft"]
    + df["total_naval_fleet"]
)

# Assets per capita
df["assets_per_capita"] = df["total_military_assets"] / df["total_population"]

# Budget to GDP ratio
df["budget_to_gdp_ratio"] = (
    df["defense_budget_usd"] / df["purchasing_power_parity_usd"]
)


RANKING KPI's

In [ ]:
# Overall power rank
df["overall_power_rank"] = df["manpower_index"].rank(
    ascending=False, method="dense"
)

# Power index rank gap
df["power_index_rank_gap"] = (
    df["manpower_potential_rank"] - df["overall_power_rank"]
)


Validate data

In [ ]:
# Check missing values %
missing_percent = df.isnull().mean() * 100
missing_percent


,0
country,0.0
total_population,0.0
total_military_manpower,0.0
fit_for_service,0.0
population_reaching_military_age_annually,0.0
...,...
total_military_assets,0.0
assets_per_capita,0.0
budget_to_gdp_ratio,0.0
overall_power_rank,0.0


Dataset for Power BI

In [ ]:

df.to_csv("military_final.csv", index=False)

print("Module 3 dataset generated successfully")


Module 3 dataset generated successfully


In [ ]:
import pandas as pd
df=pd.read_csv('military_final.csv')
df.head()

,country,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,...,land_power_index,air_power_index,naval_power_index,economic_power_index,power_category,total_military_assets,assets_per_capita,budget_to_gdp_ratio,overall_power_rank,power_index_rank_gap
0,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,1.000000,1.000000,0.657923,0.753466,High Power,410086,0.001199,0.036291,3.0,0.0
1,Russia,140820810,69002197,46189226,1267387,1320000,2000000,250000,4292,833,...,0.385600,0.345247,0.551913,0.187001,Low Power,141988,0.001008,0.021664,11.0,-2.0
2,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,0.395880,0.255607,1.000000,1.000000,High Power,154880,0.000109,0.008545,1.0,0.0
3,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,0.393567,0.164400,0.371585,0.395014,Low Power,155317,0.000110,0.005723,2.0,0.0
4,South Korea,52081799,26040900,21353538,416654,600000,3100000,120000,1592,315,...,0.173234,0.121253,0.304918,0.088055,Low Power,62935,0.001208,0.017706,22.0,6.0


In [ ]:
df.columns

Index(['country', 'total_population', 'total_military_manpower',
       'fit_for_service', 'population_reaching_military_age_annually',
       'active_personnel', 'reserve_personnel', 'paramilitary',
       'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft',
       'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft',
       'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters',
       'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
       'major_ports_and_terminals', 'total_merchant_

In [ ]:
!pip install pycountry pycountry-convert


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.5/253.5 kB 19.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import pycountry
import pycountry_convert as pc
df = pd.read_csv("military_final.csv")

# Helper functions
def get_continent(country_name):
    try:
        country_alpha2 = pycountry.countries.get(name=country_name).alpha_2
        continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
        continent_map = {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania"
        }
        return continent_map.get(continent_code, "Other")
    except:
        return "Other"

def get_region(continent):
    region_map = {
        "Asia": "Asia",
        "Europe": "Europe",
        "Africa": "Africa",
        "North America": "North America",
        "South America": "South America",
        "Oceania": "Oceania"
    }
    return region_map.get(continent, "Other")

# NATO ISO codes (official members)
nato_iso = {
    "USA","CAN","GBR","FRA","DEU","ITA","ESP","PRT","BEL","NLD","LUX",
    "NOR","DNK","ISL","POL","CZE","SVK","HUN","ROU","BGR",
    "LTU","LVA","EST","SVN","HRV","ALB","MNE","MKD","TUR","GRC"
}

# Create metadata dataframe
metadata = df[["iso_code", "country"]].drop_duplicates()

metadata["continent"] = metadata["country"].apply(get_continent)
metadata["region"] = metadata["continent"].apply(get_region)
metadata["nato_flag"] = metadata["iso_code"].apply(lambda x: "Yes" if x in nato_iso else "No")

# Save metadata file
metadata.to_csv("country_metadata.csv", index=False)

metadata.head()


,iso_code,country,continent,region,nato_flag
0,USA,United States,North America,North America,Yes
1,RUS,Russia,Other,Other,No
2,CHN,China,Asia,Asia,No
3,IND,India,Asia,Asia,No
4,KOR,South Korea,Other,Other,No


In [ ]:
metadata.columns

Index(['iso_code', 'country', 'continent', 'region', 'nato_flag'], dtype='object')

In [ ]:
df.columns

Index(['country', 'total_population', 'total_military_manpower',
       'fit_for_service', 'population_reaching_military_age_annually',
       'active_personnel', 'reserve_personnel', 'paramilitary',
       'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft',
       'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft',
       'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters',
       'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
       'major_ports_and_terminals', 'total_merchant_